In [ ]:
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


# Ajustado para o caminho exato informado
try:
    df_principal = pd.read_csv('data/Data_Streamlit/Space_Limpa_Com_Risco_Individualv2.csv')
    print("Dataset v2 (Streamlit) carregado com sucesso!")
except FileNotFoundError:
    df_principal = pd.read_csv('data/Space_Limpa_Com_Risco_Individualv1.csv')
    print("Dataset v1 carregado como fallback!")

# Remoção imediata de duplicatas do arquivo principal
df_principal = df_principal.drop_duplicates()

org_df = pd.read_csv('data/organizations.csv').drop_duplicates()
rocket_df = pd.read_csv('data/rockets.csv').drop_duplicates()
launches_df = pd.read_csv('data/launches.csv').drop_duplicates()

# Padronização interna das colunas do df_principal para evitar conflitos de espaços em branco
df_principal.columns = df_principal.columns.str.strip()

# Localizar colunas dinamicamente (independente de maiúsculas/minúsculas)
def encontrar_coluna(lista_colunas, possiveis_nomes):
    for col in lista_colunas:
        if col.lower() in [p.lower() for p in possiveis_nomes]:
            return col
    return None

col_empresa = encontrar_coluna(df_principal.columns, ['Nome_da_Empresa', 'nome_da_empresa'])
col_foguete = encontrar_coluna(df_principal.columns, ['Modelo_Foguete', 'modelo_foguete'])
col_ano = encontrar_coluna(df_principal.columns, ['Ano', 'ano'])
col_mes = encontrar_coluna(df_principal.columns, ['Mes', 'mes'])
col_status_txt = encontrar_coluna(df_principal.columns, ['Status_da_Missao', 'status_da_missao'])
col_status_num = encontrar_coluna(df_principal.columns, ['Status_da_Missao_Num', 'status_da_missao_num'])
col_prob_sucesso = encontrar_coluna(df_principal.columns, ['Probabilidade_Sucesso', 'probabilidade_sucesso'])

# Identificar dinamicamente as coordenadas (captura tanto 'latitude' quanto 'Latitude')
orig_lat = encontrar_coluna(df_principal.columns, ['latitude', 'Latitude'])
orig_long = encontrar_coluna(df_principal.columns, ['longitude', 'Longitude'])

# Guardar os valores REAIS das coordenadas se eles existirem no arquivo carregado
valores_latitude = df_principal[orig_lat].copy() if orig_lat else None
valores_longitude = df_principal[orig_long].copy() if orig_long else None

map_empresas = {
    'RVSN USSR': 'Roscosmos', 'VKS RF': 'Roscosmos', 'US Air Force': 'NASA', 
    'US Navy': 'NASA', 'Martin Marietta': 'ULA', 'Lockheed': 'ULA', 
    'Boeing': 'ULA', 'MHI': 'JAXA', 'Starsem': 'Arianespace', 'ILS': 'Roscosmos'
}
df_principal['Empresa_Ajustada'] = df_principal[col_empresa].replace(map_empresas)

def simplificar_foguete(nome_original):
    if pd.isna(nome_original): return "Não Informado"
    nome = re.sub(r'\s*\(.*?\)', '', str(nome_original))
    if '/' in nome: nome = nome.split('/')[0]
    return nome.replace('Soyuz U', 'Soyuz-U').strip()

df_principal['Foguete_Ajustado'] = df_principal[col_foguete].apply(simplificar_foguete)

# Merge 1: Organizações (Empresas)
df_mega = df_principal.merge(
    org_df[['organization', 'sector', 'founded']], 
    left_on='Empresa_Ajustada', right_on='organization', how='left'
)

# Merge 2: Foguetes (Engenharia)
df_mega = df_mega.merge(
    rocket_df[['rocket', 'propellant', 'payload_to_leo_kg', 'reusable', 'cost_per_kg_usd']], 
    left_on='Foguete_Ajustado', right_on='rocket', how='left'
)

# Merge 3: Lançamentos
launches_subset = launches_df[['year', 'month', 'rocket', 'orbit', 'mission_type', 'payload_mass_kg']].drop_duplicates(subset=['year', 'month', 'rocket'])
df_mega = df_mega.merge(
    launches_subset, 
    left_on=[col_ano, col_mes, 'Foguete_Ajustado'], right_on=['year', 'month', 'rocket'], how='left'
)

# Criar a idade da empresa na data do lançamento
df_mega['Idade_Empresa_No_Lancamento'] = df_mega[col_ano] - df_mega['founded']

print(f"Mega Merge Concluído! Total de colunas agora: {len(df_mega.columns)}")



st_colunas_eng = ['propellant', 'reusable', 'payload_to_leo_kg', 'cost_per_kg_usd', col_status_txt]
df_eng = df_mega[st_colunas_eng].dropna(subset=['propellant']).drop_duplicates().copy()

print("\n--- ANÁLISE DE ENGENHARIA ---")

taxa_combustivel = pd.crosstab(df_eng['propellant'], df_eng[col_status_txt], normalize='index') * 100
taxa_combustivel = taxa_combustivel.reset_index()
cols_y = [c for c in ['Success', 'Failure', 'sucesso', 'falha'] if c in taxa_combustivel.columns]

fig_fuel = px.bar(
    taxa_combustivel, x='propellant', y=cols_y, 
    title='Risco de Falha por Tipo de Combustível',
    labels={'value': 'Porcentagem (%)', 'propellant': 'Combustível', 'variable': 'Resultado'},
    barmode='stack', color_discrete_map={'Success': 'green', 'Failure': 'red', 'sucesso': 'green', 'falha': 'red'}
)
fig_fuel.show()

df_eng['reusable_str'] = df_eng['reusable'].map({1.0: 'Reutilizável', 0.0: 'Descartável'}).fillna('Descartável')
taxa_reuso = pd.crosstab(df_eng['reusable_str'], df_eng[col_status_txt], normalize='index') * 100

fig_reuso = px.bar(
    taxa_reuso.reset_index(), x='reusable_str', y=cols_y, 
    title='Foguetes Reutilizáveis vs Descartáveis',
    labels={'value': 'Porcentagem (%)', 'reusable_str': 'Tipo de Foguete', 'variable': 'Resultado'},
    barmode='group', color_discrete_map={'Success': 'green', 'Failure': 'red', 'sucesso': 'green', 'falha': 'red'}
)
fig_reuso.show()

corr_eng = df_mega[['payload_to_leo_kg', 'cost_per_kg_usd', col_status_num]].corr()
print("\nCorelação de Carga e Custo com o Sucesso da Missão:")
print(corr_eng[col_status_num])



df_orbita = df_mega.dropna(subset=['orbit', 'mission_type']).drop_duplicates().copy()

print("\n--- ANÁLISE DE ÓRBITA E MISSÃO ---")

col_col_fail = 'Failure' if 'Failure' in df_orbita[col_status_txt].unique() else 'falha'
taxa_orbita = pd.crosstab(df_orbita['orbit'], df_orbita[col_status_txt], normalize='index') * 100
contagem_orbita = df_orbita['orbit'].value_counts()
orbitas_principais = contagem_orbita[contagem_orbita > 5].index
taxa_orbita = taxa_orbita.loc[orbitas_principais].reset_index()

fig_orbit = px.bar(
    taxa_orbita.sort_values(by=col_col_fail, ascending=False), 
    x='orbit', y=col_col_fail, 
    title='<b>Quão perigoso é o Destino? Taxa de Falha (%) por Órbita Alvo</b>',
    labels={col_col_fail: 'Taxa de Falha (%)', 'orbit': 'Órbita Alvo'},
    color=col_col_fail, color_continuous_scale='Reds'
)
fig_orbit.show()

taxa_missao = pd.crosstab(df_orbita['mission_type'], df_orbita[col_status_txt], normalize='index') * 100
taxa_missao = taxa_missao.reset_index().sort_values(by=col_col_fail, ascending=False)

fig_mission = px.bar(
    taxa_missao, x='mission_type', y=col_col_fail, 
    title='<b>A Carga Importa? Taxa de Falha (%) por Tipo de Missão</b>',
    labels={col_col_fail: 'Taxa de Falha (%)', 'mission_type': 'Objetivo da Missão'},
    color=col_col_fail, color_continuous_scale='Oranges'
)
fig_mission.show()

corr_peso = df_mega[['payload_mass_kg', col_status_num]].corr()
print("\nCorrelação entre o Peso do Satélite (Carga) e o Sucesso da Missão:")
print(corr_peso[col_status_num])



df_corp = df_mega.dropna(subset=['sector']).drop_duplicates().copy()
if col_prob_sucesso in df_corp.columns:
    df_corp = df_corp.dropna(subset=['Idade_Empresa_No_Lancamento', col_prob_sucesso])

print("\n--- ANÁLISE CORPORATIVA TOTALMENTE CORRIGIDA ---")

taxa_setor = pd.crosstab(df_corp['sector'], df_corp[col_status_txt], normalize='index') * 100
taxa_setor = taxa_setor.reset_index()

fig_sector = px.bar(
    taxa_setor, x='sector', y=cols_y, 
    title='Taxa de Sucesso: Governo vs. Setor Privado',
    labels={'value': 'Porcentagem (%)', 'sector': 'Setor Atuação', 'variable': 'Resultado'},
    barmode='stack', color_discrete_map={'Success': 'green', 'Failure': 'red', 'sucesso': 'green', 'falha': 'red'}
)
fig_sector.show()

if col_prob_sucesso in df_corp.columns:
    fig_budget = px.scatter(
        df_corp, x='Idade_Empresa_No_Lancamento', y=col_prob_sucesso, 
        color='sector',
        hover_name='Empresa_Ajustada',
        title='Idade da Agência vs Probabilidade de Sucesso',
        labels={'Idade_Empresa_No_Lancamento': 'Anos de Existência da Empresa', col_prob_sucesso: 'Chance de Sucesso (%)', 'sector': 'Setor'}
    )
    fig_budget.update_traces(marker=dict(size=12)) 
    fig_budget.show()

corr_corp = df_mega[['Idade_Empresa_No_Lancamento', col_status_num]].corr()
print("\nCorrelação da Idade com o Sucesso da Missão:")
print(corr_corp[col_status_num])



df_limpo = df_mega.copy()

print("\n--- DIAGNÓSTICO E SALVAMENTO INICIAL ---")
print(f"Total de linhas antes da limpeza final: {len(df_limpo)}")

# Remoção profunda de duplicados antes de exportar
df_limpo = df_limpo.drop_duplicates()
print(f"Total de linhas após remover duplicatas: {len(df_limpo)}")

# Tratamento de nulos para dados numéricos de engenharia
df_limpo['payload_to_leo_kg'] = df_limpo['payload_to_leo_kg'].fillna(0)
df_limpo['cost_per_kg_usd'] = df_limpo['cost_per_kg_usd'].fillna(0)
df_limpo['payload_mass_kg'] = df_limpo['payload_mass_kg'].fillna(0)

# Dados de Texto -> Substituímos por 'Não Informado'
colunas_texto_nan = ['propellant', 'sector', 'orbit', 'mission_type']
for col in colunas_texto_nan:
    df_limpo[col] = df_limpo[col].fillna('Não Informado')

df_limpo['reusable'] = df_limpo['reusable'].fillna(0).astype(int)

# Correção da Idade da Empresa no Lançamento
mediana_idade = df_limpo['Idade_Empresa_No_Lancamento'].median()
df_limpo['Idade_Empresa_No_Lancamento'] = df_limpo['Idade_Empresa_No_Lancamento'].fillna(mediana_idade)

# --- MAPA DE TRADUÇÃO ESTÁVEL ---
mapeamento_final = {
    col_empresa: 'Nome_da_Empresa',
    'status_do_foguete': 'Status_do_Foguete',
    'custo_da_missao': 'Custo_da_Missao',
    col_status_txt: 'Status_da_Missao',
    'pais': 'Pais_Lancamento',
    'pais_lancamento': 'Pais_Lancamento',
    'estado_regiao': 'Regiao_Lancamento',
    'regiao_lancamento': 'Regiao_Lancamento',
    col_ano: 'Ano', 
    col_mes: 'Mes', 
    'dia': 'Dia', 
    'hora': 'Hora',
    col_foguete: 'Modelo_Foguete',
    'carga_util': 'Nome_Satellites_Carga', 
    'nome_satellites_carga': 'Nome_Satellites_Carga',
    col_prob_sucesso: 'Probabilidade_Sucesso_IA', 
    'probabilidade_sucesso_ia': 'Probabilidade_Sucesso_IA',
    'probabilidade_falha': 'Probabilidade_Falha_IA', 
    'probabilidade_falha_ia': 'Probabilidade_Falha_IA',
    'previsao_ia_status': 'Previsao_IA_Status',
    'sector': 'Setor_da_Empresa', 
    'setor_da_empresa': 'Setor_da_Empresa',                 
    'Idade_Empresa_No_Lancamento': 'Idade_da_Empresa', 
    'idade_da_empresa': 'Idade_da_Empresa',
    'propellant': 'Tipo_Combustivel', 
    'tipo_combustivel': 'Tipo_Combustivel',
    'payload_to_leo_kg': 'Capacidade_Maxima_LEO_KG', 
    'capacidade_maxima_leo_kg': 'Capacidade_Maxima_LEO_KG',
    'reusable': 'Foguete_Reutilizavel', 
    'foguete_reutilizavel': 'Foguete_Reutilizavel',            
    'cost_per_kg_usd': 'Custo_Por_KG_USD', 
    'custo_por_kg_usd': 'Custo_Por_KG_USD',
    'orbit': 'Orbita_Destino', 
    'orbita_destino': 'Orbita_Destino',
    'mission_type': 'Tipo_de_Missao', 
    'tipo_de_missao': 'Tipo_de_Missao',
    'payload_mass_kg': 'Peso_Real_da_Carga_KG', 
    'peso_real_da_carga_kg': 'Peso_Real_da_Carga_KG'
}

# Criamos uma função de mapeamento case-insensitive para o rename funcionar perfeitamente
mapa_normalizado = {k.lower(): v for k, v in mapeamento_final.items()}
df_limpo = df_limpo.rename(columns=lambda x: mapa_normalizado.get(x.lower(), x))

# Garante que as colunas finais traduzidas sejam selecionadas de forma correta
colunas_finais_filtradas = [v for k, v in mapeamento_final.items() if v in df_limpo.columns]
colunas_finais_filtradas = list(set(colunas_finais_filtradas)) # Remove duplicados de nomes

df_final = df_limpo[colunas_finais_filtradas].copy()

if valores_latitude is not None and valores_longitude is not None:
    df_final['Latitude'] = valores_latitude.values
    df_final['Longitude'] = valores_longitude.values
    print("Coordenadas Geográficas reais preservadas com sucesso!")
else:
    df_final['Latitude'] = 0.0
    df_final['Longitude'] = 0.0
    print("Coordenadas ausentes (v1). Inicializadas com 0.0 por segurança.")

# Preenchimento fino de remanescentes nas colunas finais
if 'Hora' in df_final.columns:
    df_final['Hora'] = df_final['Hora'].fillna('00:00')
df_final = df_final.fillna('Não Informado')

# Conversão robusta dos tipos numéricos de coordenadas e data
df_final['Latitude'] = pd.to_numeric(df_final['Latitude'], errors='coerce').fillna(0.0)
df_final['Longitude'] = pd.to_numeric(df_final['Longitude'], errors='coerce').fillna(0.0)
df_final['Dia'] = pd.to_numeric(df_final['Dia'], errors='coerce').fillna(1).astype(int)
df_final['Mes'] = pd.to_numeric(df_final['Mes'], errors='coerce').fillna(1).astype(int)
df_final['Ano'] = pd.to_numeric(df_final['Ano'], errors='coerce').fillna(1957).astype(int)

# Formatação e limpeza de textos categóricos (Title Case)
colunas_texto_ajuste = ['Setor_da_Empresa', 'Tipo_Combustivel', 'Orbita_Destino', 'Tipo_de_Missao', 'Pais_Lancamento', 'Regiao_Lancamento']
for col in colunas_texto_ajuste:
    if col in df_final.columns:
        df_final[col] = df_final[col].astype(str).str.strip().str.title()

# Remoção final de registros duplicados pós-limpeza
df_final = df_final.drop_duplicates()

print(f"\n--- DIAGNÓSTICO FINAL ---")
print(f"Colunas do DataFrame Traduzidas com Sucesso ({len(df_final.columns)} colunas):\n{df_final.columns.tolist()}")
print(f"\nTotal de valores nulos restantes no dataset: {df_final.isna().sum().sum()}")

# Salvando o dataset definitivo purificado e sem duplicatas
df_final.to_csv('data/Data_Streamlit/Space_Limpa_Com_Risco_Individualv3.csv', index=False)
print("\nO seu Dataset definitivo 'data/Data_Streamlit/Space_Limpa_Com_Risco_Individualv2.csv' foi gerado com 100% de sucesso!")


print("\n--- GERANDO DICIONÁRIO DE METADADOS INTERATIVO ---")

dicionario_mestre = {
    'Nome_da_Empresa': 'Nome da organização, empresa privada ou agência espacial governamental responsável pelo gerenciamento da missão.',
    'Status_do_Foguete': 'Estado operacional atual do modelo do veículo lançador no mercado (StatusActive para ativo ou StatusRetired para aposentado).',
    'Custo_da_Missao': 'Custo estimado do lançamento da missão em milhões de dólares americanos ($USD).',
    'Status_da_Missao': 'Resultado real verificado no momento do lançamento do veículo espacial (Success para Sucesso ou Failure para Falha).',
    'Pais_Lancamento': 'País soberano onde a base física ou plataforma de lançamento está localizada geograficamente.',
    'Regiao_Lancamento': 'Estado, província, região geográfica ou centro espacial específico de onde o foguete decolou.',
    'Ano': 'Ano civil numérico de quatro dígitos em que o lançamento espacial foi executado.',
    'Mes': 'Mês do ano ajustado e padronizado em formato numérico inteiro (1 a 12).',
    'Dia': 'Dia específico do mês em que ocorreu o lançamento (1 a 31).',
    'Hora': 'Hora aproximada em que o veículo decolou, padronizada no fuso militar de 24 horas UTC.',
    'Modelo_Foguete': 'Designação técnica descritiva e modelo específico do veículo lançador utilizado.',
    'Nome_Satellites_Carga': 'Nome de registro, sigla corporativa ou descrição detalhada da carga útil levada ao espaço.',
    'Probabilidade_Sucesso_IA': 'Probabilidade estatística de sucesso (0 a 1) calculada pelo modelo preditivo de Inteligência Artificial para a decolagem.',
    'Probabilidade_Falha_IA': 'Probabilidade estatística de falha (0 a 1) calculada pelo modelo preditivo de Inteligência Artificial para a decolagem.',
    'Previsao_IA_Status': 'Classificação binária preditiva final gerada pelo modelo de IA (1 indica sucesso previsto, 0 indica falha prevista).',
    'Latitude': 'Coordenada de latitude geográfica mapeada para a exata localização da base de lançamento.',
    'Longitude': 'Coordenada de longitude geográfica mapeada para a exata localização da base de lançamento.',
    'Setor_da_Empresa': 'Classificação institutional do modelo operacional da corporação (Commercial, Civil, Military ou Governamental).',
    'Idade_da_Empresa': 'Diferença em anos entre o ano de fundação da agência/empresa e o ano em que o lançamento ocorreu (Experiência).',
    'Tipo_Combustivel': 'Composição química e especificação técnica do propelente utilizado pelos motores do foguete (Ex: LOX/RP-1, Solid).',
    'Capacidade_Maxima_LEO_KG': 'Massa máxima em quilogramas que o veículo lançador consegue injetar na Órbita Baixa Terrestre (Engenharia).',
    'Foguete_Reutilizavel': 'Variável binária indicando se o primeiro estágio do foguete possui tecnologia de reutilização (1 para Sim, 0 para Não).',
    'Custo_Por_KG_USD': 'Indicador de eficiência financeira representando o custo em dólares cobrado por cada quilograma enviado ao espaço ($USD/Kg).',
    'Orbita_Destino': 'Tipo de órbita espacial planejada para a qual o satélite foi enviado (Leo, Geo, Gto, SSO, Meo).',
    'Tipo_de_Missao': 'Objetivo funcional e aplicação prática daquela carga útil (Comunicações, Meteorologia, Científico, GPS).',
    'Peso_Real_da_Carga_KG': 'Massa física real líquida aferida em quilogramas do satélite acoplado na coifa do veículo lançador.'
}

linhas_dados_dic = []
for col in df_final.columns:
    tipo = str(df_final[col].dtype)
    unicos = df_final[col].nunique()
    amostra = ", ".join([str(x) for x in df_final[col].dropna().unique()[:2]])
    amostra = amostra.replace('.0,', ',').replace('.0', '')
    desc = dicionario_mestre.get(col, f"Descrição não encontrada para a coluna {col}.")
    
    linhas_dados_dic.append({
        'Nome da Coluna': col,
        'Tipo do Dado': tipo,
        'Valores Únicos': unicos,
        'Exemplos Reais': amostra,
        'Descrição Detalhada': desc
    })

df_dic_dinamico = pd.DataFrame(linhas_dados_dic)

# Ordenar a tabela alfabeticamente pelo nome das colunas
df_dic_dinamico = df_dic_dinamico.sort_values(by='Nome da Coluna')

# Renderização da Tabela Interativa via Plotly Graph Objects (Sem cortes)
colunas_bold = [f"<b>{c}</b>" for c in df_dic_dinamico.columns]
total_linhas = len(df_dic_dinamico)
altura_janela = 200 + (total_linhas * 38)

fig_completo = go.Figure(data=[go.Table(
    header=dict(
        values=colunas_bold,
        fill_color='#2c3e50',
        align='left',
        font=dict(size=13, color='white', family='Arial'),
        height=38
    ),
    cells=dict(
        values=[df_dic_dinamico[c] for c in df_dic_dinamico.columns],
        fill_color=['#f8f9fa', '#ffffff'] * (total_linhas // 2 + 1),
        align='left',
        font=dict(size=11, color='#34495e', family='Arial'),
        height=35
    )
)])

fig_completo.update_layout(
    title=f"<b>📖 DICIONÁRIO DE METADADOS DINÂMICO E HIGIENIZADO — {total_linhas} COLUNAS ANALISADAS</b>",
    title_font=dict(size=15, family='Arial', color='#2c3e50'),
    width=1300,
    height=altura_janela, 
    margin=dict(l=10, r=10, t=60, b=10)
)

fig_completo.show()

Dataset v2 (Streamlit) carregado com sucesso!
Mega Merge Concluído! Total de colunas agora: 40

--- ANÁLISE DE ENGENHARIA ---



Corelação de Carga e Custo com o Sucesso da Missão:
payload_to_leo_kg       0.040142
cost_per_kg_usd        -0.057372
Status_da_Missao_Num    1.000000
Name: Status_da_Missao_Num, dtype: float64

--- ANÁLISE DE ÓRBITA E MISSÃO ---



Correlação entre o Peso do Satélite (Carga) e o Sucesso da Missão:
payload_mass_kg         0.005411
Status_da_Missao_Num    1.000000
Name: Status_da_Missao_Num, dtype: float64

--- ANÁLISE CORPORATIVA TOTALMENTE CORRIGIDA ---



Correlação da Idade com o Sucesso da Missão:
Idade_Empresa_No_Lancamento    0.058865
Status_da_Missao_Num           1.000000
Name: Status_da_Missao_Num, dtype: float64

--- DIAGNÓSTICO E SALVAMENTO INICIAL ---
Total de linhas antes da limpeza final: 4321
Total de linhas após remover duplicatas: 4321
Coordenadas Geográficas reais preservadas com sucesso!

--- DIAGNÓSTICO FINAL ---
Colunas do DataFrame Traduzidas com Sucesso (26 colunas):
['Tipo_Combustivel', 'Ano', 'Pais_Lancamento', 'Nome_Satellites_Carga', 'Dia', 'Probabilidade_Falha_IA', 'Status_do_Foguete', 'Idade_da_Empresa', 'Nome_da_Empresa', 'Capacidade_Maxima_LEO_KG', 'Foguete_Reutilizavel', 'Peso_Real_da_Carga_KG', 'Status_da_Missao', 'Previsao_IA_Status', 'Mes', 'Regiao_Lancamento', 'Tipo_de_Missao', 'Probabilidade_Sucesso_IA', 'Hora', 'Custo_da_Missao', 'Setor_da_Empresa', 'Custo_Por_KG_USD', 'Orbita_Destino', 'Modelo_Foguete', 'Latitude', 'Longitude']

Total de valores nulos restantes no dataset: 0

O seu Dataset definitiv